In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from datasetup import load_data
import torch
import torch.nn as nn
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    accuracy_score,
)
from sklearn.preprocessing import StandardScaler

def load_data():
    dfs_a = []
    for file in os.listdir("data/training/training_setA"):
        if file.endswith(".psv"):
            df = pd.read_csv(f"data/training/training_setA/{file}", sep="|")
            df["patient_id"] = file.replace(".psv", "")
            dfs_a.append(df)

    dfs_b = []
    for file in os.listdir("data/training/training_setB"):
        if file.endswith(".psv"):
            df = pd.read_csv(f"data/training/training_setB/{file}", sep="|")
            df["patient_id"] = file.replace(".psv", "")
            dfs_b.append(df)

    data = pd.concat(dfs_a + dfs_b, ignore_index=True)

    patient_ids = data["patient_id"].unique()
    train_ids, test_ids = train_test_split(patient_ids, test_size=0.2, random_state=42)
    train_ids, val_ids = train_test_split(train_ids, test_size=0.1, random_state=42)

    train = data[data["patient_id"].isin(train_ids)]
    val = data[data["patient_id"].isin(val_ids)]
    test = data[data["patient_id"].isin(test_ids)]

    return train, val, test


train, val, test = load_data()

#The features I chose for this test (can be changed later)
features = ["HR", "O2Sat", "Temp", "Resp", "Age", "ICULOS"]

#Drops missing values
train_clean = train[features + ["SepsisLabel"]].dropna()
val_clean = val[features + ["SepsisLabel"]].dropna()
test_clean = test[features + ["SepsisLabel"]].dropna()

X_train = train_clean[features].values
y_train = train_clean["SepsisLabel"].values

X_val = val_clean[features].values
y_val = val_clean["SepsisLabel"].values

X_test = test_clean[features].values
y_test = test_clean["SepsisLabel"].values

#Scales the inputs
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

#Converts the data into tensor form
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(len(features), 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}/{epochs} | Loss: {loss.item():.4f}")

#Here are the metrics for validation
with torch.no_grad():
    val_logits = model(X_val)
    val_probs = torch.sigmoid(val_logits).numpy().flatten()
    val_preds = (val_probs >= 0.5).astype(int)

print("\nNeural Network Metrics Validation")
print(f"Accuracy:  {accuracy_score(y_val, val_preds):.4f}")
print(f"Precision: {precision_score(y_val, val_preds):.4f}")
print(f"Recall:    {recall_score(y_val, val_preds):.4f}")
print(f"F1 Score:  {f1_score(y_val, val_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, val_probs):.4f}")
print("\nFull Report:")
print(classification_report(y_val, val_preds, target_names=["No Sepsis", "Sepsis"]))
print("Confusion Matrix:")
print(confusion_matrix(y_val, val_preds))

#Here are the metrics for Testing
with torch.no_grad():
    test_logits = model(X_test)
    test_probs = torch.sigmoid(test_logits).numpy().flatten()
    test_preds = (test_probs >= 0.5).astype(int)

print("\nNeural Network Metrics Test")
print(f"Accuracy:  {accuracy_score(y_test, test_preds):.4f}")
print(f"Precision: {precision_score(y_test, test_preds):.4f}")
print(f"Recall:    {recall_score(y_test, test_preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, test_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, test_probs):.4f}")
print("\nFull Report:")
print(classification_report(y_test, test_preds, target_names=["No Sepsis", "Sepsis"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_preds))